In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd
import numpy as np
from src.hypothesis_tests import t_test, chi_square_test, decision

In [3]:
df = pd.read_csv("../data/insurance_data_cleaned.csv")
df.head()

,customerid,age,gender,province,vehicletype,annualincome,riskscore,annualpremium,deductible,ncd,...,claimed,claimamount,totalpremium,totalclaims,covertype,automake,vehiclemodel,customvalueestimate,zipcode,transactiondate
0,AC-100000,56,Male,Addis Ababa,Sedan,147270,61,2346,500,30,...,False,0.0,2346,0.0,Comprehensive,Lifan,620,32238,10002,2024-05-10
1,AC-100001,69,Female,Addis Ababa,SUV,74640,57,2334,500,0,...,True,9883.0,2334,9883.0,Comprehensive,Suzuki,Grand Vitara,52510,10001,2024-08-13
2,AC-100002,46,Male,Oromia,Sedan,70555,42,1697,250,20,...,False,0.0,1697,0.0,Third Party Fire & Theft,Lifan,620,26523,20001,2025-03-17
3,AC-100003,32,Female,Somali,Sedan,89398,63,2370,500,20,...,True,12134.0,2370,12134.0,Comprehensive,Toyota,Corolla,27036,40005,2025-03-17
4,AC-100004,60,Female,Tigray,SUV,78475,69,2582,500,0,...,False,0.0,2582,0.0,Comprehensive,Toyota,RAV4,58348,50002,2024-11-10


In [4]:
#KPI DEFINITIONS
df["has_claim"] = df["totalclaims"] > 0

claim_frequency = df["has_claim"].mean()


In [5]:
#CLAIM SEVERITY
claim_severity = df[df["has_claim"]]["totalclaims"].mean()

In [6]:
#MARGIN
df["margin"] = df["totalpremium"] - df["totalclaims"]

In [19]:
#HYPOTHESIS 1
table = pd.crosstab(df["province"], df["has_claim"])

stat, p_h1 = chi_square_test(table)
decision_h1 = decision(p_h1)

p_h1

np.float64(0.07608907812609692)

In [20]:
#HYPOTHESIS 2: Zip codes → Risk differences
group_a = df[df["zipcode"] == "A"]["totalclaims"]
group_b = df[df["zipcode"] == "B"]["totalclaims"]

stat, p_h2 = t_test(group_a, group_b)
decision_h2 = decision(p_h2)

p_h2

d:\KAIM\insurance-risk-analytics\src\hypothesis_tests.py:8: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  stat, p_value = stats.ttest_ind(group_a, group_b, equal_var=False)


np.float64(nan)

In [21]:
#HYPOTHESIS 3: Zip codes → Margin
group_a = df[df["zipcode"] == "A"]["margin"]
group_b = df[df["zipcode"] == "B"]["margin"]

stat, p_h3 = t_test(group_a, group_b)
decision_h3 = decision(p_h3)

p_h3

p

np.float64(0.9638306173980254)

In [22]:
#HYPOTHESIS 4: Gender → Risk difference
table = pd.crosstab(df["gender"], df["has_claim"])

stat, p_h4 = chi_square_test(table)
decision_h4 = decision(p_h4)

p_h4

np.float64(0.9638306173980254)

In [23]:
#BUILD RESULTS TABLE
results = pd.DataFrame([
    ["Province vs Risk", "Claim Frequency", p_h1, decision_h1],
    ["Zipcode vs Severity", "Claim Severity", p_h2, decision_h2],
    ["Zipcode vs Margin", "Margin", p_h3, decision_h3],
    ["Gender vs Risk", "Claim Frequency", p_h4, decision_h4]
], columns=["Hypothesis", "KPI", "p-value", "Decision"])

results

,Hypothesis,KPI,p-value,Decision
0,Province vs Risk,Claim Frequency,0.076089,Fail to Reject H0
1,Zipcode vs Severity,Claim Severity,NaN,Fail to Reject H0
2,Zipcode vs Margin,Margin,NaN,Fail to Reject H0
3,Gender vs Risk,Claim Frequency,0.963831,Fail to Reject H0
